# Experiment 5.4.1 — constrained WHAT×WHEN residual conjunction

Analysis-only notebook for `constrained_conjunction_residual_v1`. The primary question is whether frozen WHEN spikes add incremental class value when they can only correct a frozen Exp5.4 WHAT-only classifier through a structural spike conjunction.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / 'notebooks').is_dir():
    repo_root = repo_root.parent
root = repo_root / 'notebooks' / 'artifacts' / 'experiment_5_4_1_constrained_conjunction_residual' / 'constrained_conjunction_residual_v1'
runs = pd.read_csv(root / 'runs.csv')
histories = pd.read_csv(root / 'histories.csv')
ablations = pd.read_csv(root / 'ablation_runs.csv')
activity = pd.read_csv(root / 'activity_runs.csv')
sources = pd.read_csv(root / 'source_runs.csv')
paired = pd.read_csv(root / 'paired_deltas.csv')
manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
manifest


## Incremental value over the frozen WHAT-only base

The primary metric is the paired per-seed test balanced-accuracy delta relative to the exact Exp5.4 `what_only_lif` checkpoint used inside the model.


In [ ]:
summary = pd.DataFrame({
    'test_ba_mean': [runs['test_balanced_accuracy'].mean()],
    'test_ba_sd': [runs['test_balanced_accuracy'].std()],
    'base_ba_mean': [runs['source_base_test_balanced_accuracy'].mean()],
    'delta_vs_base_mean': [runs['delta_test_ba_vs_base'].mean()],
    'delta_vs_base_sd': [runs['delta_test_ba_vs_base'].std()],
    'wins_vs_base': [int((runs['delta_test_ba_vs_base'] > 0).sum())],
    'epoch0_selected': [int((runs['best_epoch'] == 0).sum())],
})
summary


In [ ]:
paired_summary = (
    paired.groupby('comparator', as_index=False)
    .agg(
        delta_ba_mean=('delta_test_balanced_accuracy', 'mean'),
        delta_ba_sd=('delta_test_balanced_accuracy', 'std'),
        wins=('delta_test_balanced_accuracy', lambda x: int((x > 0).sum())),
        delta_macro_f1_mean=('delta_test_macro_f1', 'mean'),
    )
    .sort_values('delta_ba_mean', ascending=False)
)
paired_summary


## History/alignment attribution and structural shortcut checks

`when_zero` and `base_context_zero` are hard structural controls: their residual correction should be exactly zero, so their final predictions should match the frozen base. The informative temporal tests are ordered vs reset, shuffled, and circularly shifted WHEN.


In [ ]:
ablation_summary = (
    ablations.groupby('ablation', as_index=False)
    .agg(
        test_ba_mean=('test_balanced_accuracy', 'mean'),
        test_ba_sd=('test_balanced_accuracy', 'std'),
        residual_abs_mean=('residual_mean_abs_logit', 'mean'),
        conjunction_fr_mean=('conjunction_firing_fraction', 'mean'),
    )
    .sort_values('test_ba_mean', ascending=False)
)
ablation_summary


In [ ]:
ordered = ablations[ablations['ablation'] == 'ordered'].set_index('seed')
alignment_rows = []
for name in ('reset_when', 'when_shuffle', 'when_circular_shift'):
    control = (
        ablations[ablations['ablation'] == name]
        .groupby('seed')['test_balanced_accuracy'].mean()
    )
    delta = ordered['test_balanced_accuracy'] - control
    alignment_rows.append({
        'ablation': name,
        'aligned_minus_ablation_ba_mean': delta.mean(),
        'aligned_minus_ablation_ba_sd': delta.std(),
        'aligned_wins': int((delta > 0).sum()),
    })
alignment = pd.DataFrame(alignment_rows)
alignment


In [ ]:
zero_checks = ablations[ablations['ablation'].isin(['when_zero', 'base_context_zero'])]
zero_checks.groupby('ablation', as_index=False).agg(
    residual_abs_max=('residual_mean_abs_logit', 'max'),
    conjunction_fr_max=('conjunction_firing_fraction', 'max'),
    test_ba_mean=('test_balanced_accuracy', 'mean'),
)


## Training dynamics and residual activity


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for seed, group in histories.groupby('seed'):
    ax.plot(group['epoch'], group['val_balanced_accuracy'], label=f'seed {seed}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation balanced accuracy')
ax.set_title('Exp5.4.1 validation BA')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()


## Interpretation checklist

The strongest support is: positive paired BA vs the exact frozen base, ordered > reset/shuffle/shift, and exact zero residual under either missing conjunction side. If epoch 0 is selected for most seeds, the constrained WHEN correction did not add reliable validation value and the correct result is the unchanged base.
